# 02 · Tabular benchmark (paper §4.1)

METABRIC and SUPPORT, the two datasets of SurvTRACE's Table 2, under its protocol:
10 random 60/10/30 splits, C_td (IPCW, censoring from training) at the 25/50/75% event-time
quantiles, mean (std). Our model uses L_PCH (recipe S1); the loss ablations are in 05.

Models: ours, Cox, Random Survival Forest, DeepSurv, PC-Hazard, DeepHit, DSM, MENSA and
SurvTRACE (authors' released code), all with the settings tuned in notebook 01.

## Options

In [ ]:
NAME = "02_tabular_benchmark"

SEEDS = list(range(10))   # protocol: 10 random 60/10/30 splits (as SurvTRACE); seed s = split s and init s

MODELS = ["ours", "cox", "rsf", "deepsurv", "pchazard", "deephit", "dsm", "mensa", "survtrace"]

DATASETS = ["metabric", "support"]
# published averages over 25/50/75% (SurvTRACE Table 2), for the report
PUBLISHED = {"metabric": {"SurvTRACE (paper)": 0.691}, "support": {"SurvTRACE (paper)": 0.640}}

# --- execution ------------------------------------------------------------------------
SMOKE_TEST   = False    # True: 2 epochs, 1 seed, 1-2 configurations - only to check that it runs
SMOKE_EPOCHS = 2
WORKERS      = None     # parallel runs; None = 2 on Kaggle (4 vCPU), 4 locally
TIME_BUDGET_MIN = 11 * 60   # stop launching new runs after this (Kaggle sessions end at 12 h);
                            # re-run with this notebook's output attached to continue
INSTALL_DEPS = True     # Kaggle only
GIT_URL      = "https://github.com/Parsagh05/dmmst.git"   # the code is cloned from here
CLEAN_UP_AT_END = True  # Kaggle: delete the working repo copy after the report

## Setup (identical in every notebook)

In [ ]:
# ---- 1. get the code --------------------------------------------------------------
import os, sys, json, shutil, subprocess, time, hashlib
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").is_dir()

def _is_repo(p):
    return (p / "sat" / "finetune.py").is_file() and (p / "conf").is_dir()

if ON_KAGGLE:
    WORK = Path("/kaggle/working")
    REPO = WORK / "dmmst"
    if not _is_repo(REPO):
        subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(REPO)], check=True)
    RESULTS_ROOT = WORK / "results"
    INPUT_ROOTS = [Path("/kaggle/input")]
else:
    REPO = Path.cwd().resolve()
    while not _is_repo(REPO) and REPO != REPO.parent:
        REPO = REPO.parent
    RESULTS_ROOT = REPO / "results" / "version_2"
    INPUT_ROOTS = []
assert _is_repo(REPO), REPO
os.chdir(REPO)
sys.path.insert(0, str(REPO))
CODE_FINGERPRINT = hashlib.md5(b"".join(
    p.read_bytes() for p in sorted((REPO / "sat").rglob("*.py")) + sorted((REPO / "conf").rglob("*.yaml"))
)).hexdigest()[:10]
COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO, text=True,
                        capture_output=True).stdout.strip()
print("repo:", REPO, "| commit:", COMMIT, "| code fingerprint:", CODE_FINGERPRINT)

In [ ]:
# ---- 2. dependencies (Kaggle) ------------------------------------------------------
# numpy and torch are NEVER moved: pandas/pyarrow and CUDA in the image are built for them.
if ON_KAGGLE and INSTALL_DEPS:
    import numpy as _np, torch
    _tv = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
    pkgs = ["transformers==4.50.0", "datasets==3.4.1", "tokenizers>=0.21,<0.22", "evaluate>=0.4.3",
            "accelerate>=1.4.0", "hydra-core>=1.3.2", "hydra-colorlog>=1.2.0",
            "torchsurv" if _tv >= (2, 8) else "torchsurv==0.1.4",
            "h5py>=3.13", "logdecorator>=2.5", "einops==0.8.1", "torchtuples>=0.2.2",
            "numba", "nvidia-ml-py", "polars", "tqdm", "tabulate",
            "scikit-survival" if int(_np.__version__.split(".")[0]) >= 2 else "scikit-survival==0.22.2",
            "pycox==0.3.0", "peft", "easydict", "lifelines", "osqp", "patsy",
            f"numpy=={_np.__version__}", f"torch=={torch.__version__.split('+')[0]}"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
    # SurvivalEVAL declares torch>=2.13; it runs on older torch, so its pins are skipped
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "SurvivalEVAL==0.8.5"], check=True)
import torch
print("torch", torch.__version__, "| cuda" if torch.cuda.is_available() else "| cpu")

In [ ]:
# ---- 3. runner ----------------------------------------------------------------------
from IPython.display import Markdown, display
import pandas as pd
from scripts.runner import Runner, select_best
from scripts import v2

RUN_NAME = NAME + ("_smoke" if SMOKE_TEST else "")
r = Runner(repo=REPO, results_dir=RESULTS_ROOT / RUN_NAME,
           workers=WORKERS or (2 if ON_KAGGLE else 4),
           smoke_epochs=SMOKE_EPOCHS if SMOKE_TEST else None, gpu="0")
r.commit = COMMIT

# Resume: attach this notebook's previous output ("Add Input") and finished runs are
# copied back, so only the missing ones are trained.
def _incomplete(f):
    """Results scored by code that has since been fixed - rerun them:
    * survival results without IBS (IBS/IBLL/AUC fixes 9b048a8, bae0c14);
    * hsa_synthetic models scored by the trainer (script "finetune") before 070f18b,
      whose Brier dropped the ~70% of subjects censored at the end of follow-up.
      Results from 070f18b on record the commit in their .meta.json."""
    if f.name.endswith(".meta.json") or f.name == "environment.json" or not f.is_file():
        return False
    test = json.loads(f.read_text()).get("test", {})
    if "ctd_weighted_avg" in test and "ibs" not in test:
        return True
    meta_f = f.with_name(f.name[:-len(".json")] + ".meta.json")
    if "ctd_weighted_avg" in test and meta_f.is_file():
        meta = json.loads(meta_f.read_text())
        if (meta.get("info", {}).get("dataset") == "hsa_synthetic" and meta.get("script") == "finetune"
                and not meta.get("commit")):
            return True
    return False

n_copied, n_redo = 0, 0
for root in INPUT_ROOTS:
    for d in root.rglob(RUN_NAME):
        if d.is_dir() and d != r.results_dir:
            for f in d.glob("*.json"):
                if (r.results_dir / f.name).exists():
                    continue
                if _incomplete(f) or _incomplete(d / f.name.replace(".meta.json", ".json")):
                    n_redo += f.name.endswith(".meta.json") is False
                    continue
                shutil.copy2(f, r.results_dir / f.name); n_copied += 1
print(f"resumed {n_copied} files from attached inputs; {n_redo} runs without IBS will be rerun")
(r.results_dir / "environment.json").write_text(json.dumps({
    "commit": COMMIT, "code_fingerprint": CODE_FINGERPRINT, "torch": torch.__version__,
    "cuda": torch.cuda.is_available(), "smoke": SMOKE_TEST, "name": RUN_NAME}, indent=2))
print("results ->", r.results_dir)

In [ ]:
# ---- settings tuned on validation in notebook 01 -------------------------------------
# an attached 01 output wins; otherwise the copy committed to GitHub
# (results/version_2/01_tuning/tuned.json in the cloned repo)
TUNED_PATH = v2.find_tuned(RESULTS_ROOT, INPUT_ROOTS + [REPO / "results" / "version_2" / "01_tuning"])
print("tuned settings from", TUNED_PATH)
TUNED = v2.load_tuned(TUNED_PATH)
LOSS = TUNED["losses"]
print(json.dumps(TUNED, indent=1)[:1500])

In [ ]:
def tuned(model, name):
    return v2.tuned_overrides(TUNED, model, name)

def loss_extra(name):
    """Per-dataset extras: time unit chosen in notebook 01."""
    return ["v2_time_scale=1.0"] if LOSS[name].get("time_scale_raw") else []

def surv_recipes(name):
    L = LOSS[name]
    multi = v2.DATASETS[name]["events"] > 1
    return v2.survival_recipes(multi, L["rank"], L.get("mul", {"coeff": 0, "sigma": 0.1}))

def reg_recipes(name):
    L = LOSS[name]
    return v2.regression_recipes(v2.DATASETS[name]["events"] > 1, L.get("mm_coeff", 0.5))

## Runs

In [ ]:
if SMOKE_TEST:
    SEEDS, MODELS = SEEDS[:1], ["ours", "cox", "survtrace"]
v2.prepare(r, DATASETS, SEEDS, REPO)
runs = []
for name in DATASETS:
    for model in MODELS:
        runs += v2.runs(model, name, SEEDS, REPO, extra=tuned(model, name), info={"stage": "benchmark"})
print(len(runs), "runs")
r.run_many(runs, time_budget_min=TIME_BUDGET_MIN)

## SurvTRACE-format table (C_td at 25/50/75%) and all metrics

In [ ]:
# ---- tables + zip ---------------------------------------------------------------------
rep = r.results_dir / "report"
rep.mkdir(exist_ok=True)
res = r.collect()
res.to_csv(rep / "all_runs.csv", index=False)
if len(res):
    t = v2.table(res, ["dataset", "model", "variant"])
    t.to_csv(rep / "summary.csv", index=False)
    display(t)
print("zip:", r.zip())

ht = v2.horizon_table(res, ["dataset", "model"])
ht.to_csv(rep / "ctd_horizons.csv", index=False)
display(ht)

## Clean up

In [ ]:
if ON_KAGGLE and CLEAN_UP_AT_END:
    shutil.rmtree(REPO, ignore_errors=True)
    print("removed", REPO, "- results are in", r.results_dir)